In [1]:
# Cell 1: Hardware Inspection
import subprocess, os, shutil

print("=== GPU Hardware Check ===")
!nvidia-smi --query-gpu=index,name,memory.total,memory.free,driver_version,pci.bus_id --format=csv

print("\n=== System RAM & CPU Check ===")
!free -h
!lscpu | grep -E "Model name|CPU\(s\):|Thread\(s\) per core|Core\(s\) per socket"

print("\n=== CUDA & Compiler Check ===")
!nvcc --version || echo "nvcc not in path, searching..."
if os.path.exists("/usr/local/cuda/bin/nvcc"):
    os.environ["PATH"] = f"/usr/local/cuda/bin:{os.environ['PATH']}"
    os.environ["LD_LIBRARY_PATH"] = f"/usr/local/cuda/lib64:{os.environ.get('LD_LIBRARY_PATH', '')}"
    !nvcc --version

In [ ]:
# Cell 2: Clone Strata & Setup Python Environment
import os

%cd /kaggle/working

# Clone Strata
if not os.path.exists("Strata"):
    !git clone https://github.com/eddoursul/Strata.git || git clone https://github.com/Niko1221/Strata.git
    %cd Strata
    # Attempt to checkout the 'custom' branch if it exists, otherwise stay on main
    !git checkout custom 2>/dev/null || echo "Branch 'custom' not found, using $(git branch --show-current)"
else:
    %cd Strata

# Install dependencies for high-speed downloads, model packing, and API testing
!pip install -q huggingface_hub aria2 numpy requests openai


In [ ]:
# Cell 3: CMake Configuration and Compilation for sm_75 (Turing)
import os, multiprocessing

%cd /kaggle/working/Strata

# Set CUDA paths
os.environ["CUDA_HOME"] = "/usr/local/cuda"
os.environ["PATH"] = f"/usr/local/cuda/bin:{os.environ['PATH']}"
os.environ["LD_LIBRARY_PATH"] = f"/usr/local/cuda/lib64:{os.environ.get('LD_LIBRARY_PATH', '')}"

num_jobs = multiprocessing.cpu_count()
print(f"Building Strata targeting CUDA sm_75 using {num_jobs} parallel threads...")

# Configure with CMake explicitly specifying sm_75 for Tesla T4
!cmake -B build -S . \
  -DCMAKE_BUILD_TYPE=Release \
  -DCMAKE_CUDA_ARCHITECTURES=75 \
  -DSTRATA_BUILD_TESTS=OFF

# Compile Release binary
!cmake --build build --config Release -j{num_jobs}

# Verify binary
!./build/strata --version 2>/dev/null || ./build/strata --help | head -n 10


In [ ]:
# Cell 4: Download 2-Shard GGUF Model from Hugging Face
import os
from huggingface_hub import hf_hub_download

MODELS_DIR = "/kaggle/working/models"
os.makedirs(MODELS_DIR, exist_ok=True)

# Target Model: Qwen 3.8 Flash IQ3_S (or IQ3_XXS) across 2 shards:
# Shard 1 holds main weights & dense layers; Shard 2 holds the PLE / experts shard
REPO_ID = "ISTA-DASLab/Qwen3.8-Flash-Next-GSQ-RCO-GGUF"  # or ukisai/Swift-1.5-Qwen3.8-Flash-Next-GSQ-RCO-GGUF
SHARDS = [
    "Qwen3.8-Flash-Next-GSQ-RCO-IQ3_S-00001-of-00002.gguf",
    "Qwen3.8-Flash-Next-GSQ-RCO-IQ3_S-00002-of-00002.gguf"
]

print(f"Downloading 2 shards from {REPO_ID} to {MODELS_DIR}...")
for shard in SHARDS:
    dest_path = os.path.join(MODELS_DIR, shard)
    if os.path.exists(dest_path):
        print(f"✓ Shard already exists: {shard} ({os.path.getsize(dest_path)/1e9:.2f} GB)")
    else:
        print(f"--> Downloading {shard}...")
        hf_hub_download(
            repo_id=REPO_ID,
            filename=shard,
            local_dir=MODELS_DIR,
            local_dir_use_symlinks=False
        )
        print(f"✓ Downloaded {shard} successfully!")

!ls -lh {MODELS_DIR}


In [ ]:
# Cell 5: Generate Native Pack & Extract Tokenizer
import os

%cd /kaggle/working/Strata

MODELS_DIR = "/kaggle/working/models"
PACK_DIR = "/kaggle/working/packs/iq3_s"
SHARD1 = os.path.join(MODELS_DIR, "Qwen3.8-Flash-Next-GSQ-RCO-IQ3_S-00001-of-00002.gguf")

os.makedirs(PACK_DIR, exist_ok=True)

# iq_pack.py inspects both shards, writes a tiny dense pack (~1.4 GB) and exports tokenizer
print("Generating native pack from GGUF shards...")
!python tools/iq_pack.py \
  --gguf "{SHARD1}" \
  --out "{PACK_DIR}" \
  --compat-bf16

!ls -lh "{PACK_DIR}"


In [ ]:
# Cell 6: Write Optimized Dual-GPU Configuration
import json, os

config = {
    "exe": "/kaggle/working/Strata/build/strata",
    "args": [
        "--pack", "/kaggle/working/packs/iq3_s",
        "--native", "/kaggle/working/models/Qwen3.8-Flash-Next-GSQ-RCO-IQ3_S-00001-of-00002.gguf",
        "--ple-gguf", "/kaggle/working/models/Qwen3.8-Flash-Next-GSQ-RCO-IQ3_S-00002-of-00002.gguf",
        "--expert-profile", "/kaggle/working/Strata/data/expert-profile.bin",
        "--expert-cache", "auto",
        "--prefill", "auto",
        "--spec", "4",
        "--max-context", "32768",
        "--kv", "int8",
        "--resident-experts",
        "--resident-pin",
        "--adapt-every", "64",
        "--adapt-swaps", "64",
        "--vram-reserve-mib", "400",
        "--vram-reserve-later-mib", "400"
    ],
    "cwd": "/kaggle/working/Strata",
    "tokenizer": "/kaggle/working/packs/iq3_s/tokenizer",
    "model_name": "qwen3.8-flash-next-iq3_s",
    "log": "/kaggle/working/strata-server.log",
    "lib_dirs": [
        "/usr/local/cuda/bin",
        "/usr/local/cuda/lib64"
    ],
    "port": 8080,
    "gpu": [0, 1],
    "gpus_asked": True,
    "layer_split": "auto",
    "host": "127.0.0.1"
}

with open("/kaggle/working/Strata/strata-2xt4.json", "w") as f:
    json.dump(config, f, indent=2)

print("✓ Created /kaggle/working/Strata/strata-2xt4.json")


In [ ]:
# Cell 7: Standalone Generation Verification (Testing Complement Allocation & Zero-Copy Rotation)
import os

%cd /kaggle/working/Strata

# Set Production Environment Variables
os.environ["STRATA_STAGE_TRIM"] = "1"        # Saves ~3.4 GB VRAM per card
os.environ["STRATA_EXCHANGE_ROTATE"] = "1"   # PR #864 zero-copy cyclic buffer rotation
os.environ["STRATA_PREFILL_HELP"] = "1"      # Idle GPU prefill assistance
os.environ["STRATA_PIPELINE_ADAPT_ASYNC"] = "1"

print("Running test prompt generation across both T4 GPUs...")
!./build/strata generate \
  --pack /kaggle/working/packs/iq3_s \
  --native /kaggle/working/models/Qwen3.8-Flash-Next-GSQ-RCO-IQ3_S-00001-of-00002.gguf \
  --ple-gguf /kaggle/working/models/Qwen3.8-Flash-Next-GSQ-RCO-IQ3_S-00002-of-00002.gguf \
  --expert-profile /kaggle/working/Strata/data/expert-profile.bin \
  --gpus 0,1 \
  --layer-split auto \
  --resident-experts \
  --resident-pin \
  --expert-cache auto \
  --adapt-every 64 \
  --adapt-swaps 64 \
  --vram-reserve-mib 400 \
  --vram-reserve-later-mib 400 \
  --prompt "Explain quantum computing in simple terms." \
  --max-tokens 128


In [ ]:
# Cell 8: Launch Strata Background Server (OpenAI Compatible)
import subprocess, time, requests, os

%cd /kaggle/working/Strata

env = os.environ.copy()
env["STRATA_STAGE_TRIM"] = "1"
env["STRATA_EXCHANGE_ROTATE"] = "1"
env["STRATA_PREFILL_HELP"] = "1"

log_file = open("/kaggle/working/strata-server.log", "w")

print("Starting Strata Server in background on port 8080...")
proc = subprocess.Popen(
    ["python", "-m", "serve.server", "--engine", "strata", "--config", "strata-2xt4.json", "--port", "8080"],
    stdout=log_file,
    stderr=subprocess.STDOUT,
    env=env
)

print("Waiting for model loading (allocating ~22 GB in VRAM and ~26 GB complement in RAM)...")
for i in range(60):
    time.sleep(2)
    try:
        r = requests.get("http://127.0.0.1:8080/health", timeout=1)
        if r.status_code == 200:
            print("\n✓ Strata Server is READY and listening on http://127.0.0.1:8080!")
            break
    except Exception:
        print(".", end="", flush=True)

# Inspect memory usage
!nvidia-smi
!free -h


In [ ]:
# Cell 9: Prefill & Decode Tok/s Benchmark
import requests, time, json

URL = "http://127.0.0.1:8080/v1/chat/completions"

def benchmark_request(prompt, max_tokens=128):
    payload = {
        "model": "qwen3.8-flash-next-iq3_s",
        "messages": [{"role": "user", "content": prompt}],
        "max_tokens": max_tokens,
        "stream": True
    }
    
    t0 = time.perf_counter()
    resp = requests.post(URL, json=payload, stream=True)
    
    first_token_time = None
    token_count = 0
    full_text = ""
    
    for line in resp.iter_lines():
        if line:
            line = line.decode('utf-8')
            if line.startswith("data: ") and line != "data: [DONE]":
                if first_token_time is None:
                    first_token_time = time.perf_counter()
                chunk = json.loads(line[6:])
                delta = chunk["choices"][0]["delta"].get("content", "")
                full_text += delta
                token_count += 1
                
    t_end = time.perf_counter()
    ttft = (first_token_time - t0) if first_token_time else 0
    decode_time = (t_end - first_token_time) if first_token_time else 0
    tok_per_sec = (token_count / decode_time) if decode_time > 0 else 0
    
    print(f"=== Benchmark Results ===")
    print(f"Prompt length: {len(prompt)} chars")
    print(f"Generated tokens: {token_count}")
    print(f"Time to First Token (TTFT): {ttft*1000:.2f} ms")
    print(f"Decode Generation Speed: {tok_per_sec:.2f} tokens/sec")
    print(f"Total Response Time: {t_end - t0:.2f} s")
    print("---------------------------------")
    print("Response Preview:\n", full_text[:200] + "...\n")

print("Running Warmup...")
benchmark_request("Hello! Introduce yourself in one sentence.", max_tokens=32)

print("\nRunning Standard Generation Benchmark (128 tokens)...")
benchmark_request("Write an essay on the architecture of large language models and mixture of experts.", max_tokens=128)
